In [0]:
import sys
sys.path.append("..")

from ingestion.utils.config_manager import ConfigManager, SOURCE_SYSTEM_TABLE, CONFIG_MASTER_TABLE
from ingestion.utils.secrets import SecretResolver

In [0]:
source_name = dbutils.widgets.get("source_name").strip()

print(f"Fetching configuration for source: {source_name}...")

query = f"""
    SELECT 
        source_id,
        source_name,
        source_type,
        host,
        port,
        database_name,
        driver_class,
        secret_scope,
        secret_key_credentials,
        source_id,
        Data_Dictionary_Enable,
        landing_volume_path
    FROM migration_x_catalog.pfl_x_schema.config_source_system 
    WHERE source_name = '{source_name}' and is_active = 1
"""
config_df = spark.sql(query).collect()

if not config_df:
    raise ValueError(f"No config found for source_name: {source_name}")

source_dict = config_df[0].asDict()
s3_bucket = str(source_dict["landing_volume_path"]).strip() if source_dict["landing_volume_path"] else ""

In [0]:
nb_path = f"/Workspace/Shared/pfl-ingestion-framework-dd-check/databricks-ingestion-framework/src/dd_validation/get_all_table_data_dictionary"
# TODO - Create the data_dictionary folder and data_dictionary file inside the PFL/Admin/Config/Data_Dictionary/get_all_table_data_dictionary

# First check whether the data_dictionary folder is there or not. If it is not there then create it.
dd_folder_name = "data_dictionary"
dd_folder_path = f"{s3_bucket}/{dd_folder_name}"
dd_file_name = source_name + "_source_data_dictionary.parquet"

raw_s3_path = f"{s3_bucket}/{dd_folder_name}/{dd_file_name}"

# Validate required config values
if not nb_path:
    raise ValueError("notebook_path is empty in config. Cannot determine which notebook returns the DD query.")
if not s3_bucket or not dd_folder_name or not dd_file_name:
    raise ValueError("S3 destination config incomplete. Check Raw_S3_Bucket_Name, Raw_DD_Folder, Raw_DD_File_Name.")

# Construct the target S3 path
raw_s3_path = f"{s3_bucket}/{dd_folder_name}/{dd_file_name}"
print(f"Target S3 path: {raw_s3_path}")

In [0]:
source_query = dbutils.widgets.get("source_query")

if not source_query or not source_query.strip():
    raise ValueError(f"Notebook '{nb_path}' did not return a valid query.")

print(f"Query retrieved successfully:\n{source_query}")

In [0]:
# STEP 2: Connect to source system & execute the query using existing framework
print(f"\nStep 2: Connecting to source system and executing query...")

# Resolve credentials from the source system config
secrets = SecretResolver(dbutils)
username, password = secrets.get_credentials(
    source_dict["secret_scope"],
    source_dict["secret_key_credentials"]
)

# Build JDBC connection options using the framework's URL builder pattern
from ingestion.connectors.jdbc_connector import _build_url, _DEFAULT_DRIVER

source_type = str(source_dict["source_type"]).upper()
host = source_dict["host"]
port = int(source_dict["port"]) if source_dict["port"] else 0
database_name = source_dict["database_name"] or ""

jdbc_url = _build_url(
    source_type=source_type,
    host=host,
    port=port,
    database_name=database_name,
    extra_params={}
)

driver = source_dict["driver_class"] if source_dict["driver_class"] else _DEFAULT_DRIVER.get(source_type)
if not driver:
    raise ValueError(f"No JDBC driver found for source_type='{source_type}'. Set driver_class in config.")

print(f"JDBC URL: {jdbc_url}")
print(f"Driver: {driver}")

# Execute the Data Dictionary query against the source system
try:
    raw_df = (spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("user", username)
        .option("password", password)
        .option("driver", driver)
        .option("query", source_query)
        .load())

    row_count = raw_df.count()
    print(f"Query returned {row_count} rows.")

except Exception as e:
    raise RuntimeError(f"JDBC extraction failed for source '{source_name}': {str(e)}")


# STEP 2.1: Write results to the configured S3 location
print(f"\nStep 3: Writing {row_count} rows to {raw_s3_path}...")

try:
    raw_df.write.format("parquet").mode("overwrite").save(raw_s3_path)
    print(f"Data Dictionary data successfully written to: {raw_s3_path}")
except Exception as e:
    raise RuntimeError(f"Failed to write Data Dictionary to S3: {str(e)}")


In [0]:
dbutils.jobs.taskValues.set(key="s3_bucket", value=s3_bucket)
dbutils.jobs.taskValues.set(key="dd_file_name", value=dd_file_name)
